# 04 - XLM-RoBERTa fine-tuning (run on Google Colab)

**Before running:** Runtime -> Change runtime type -> **T4 GPU** -> Save.

Experiments (same Roman Urdu test set as Weeks 2 and 3, so numbers are comparable):
1. Zero-shot: train on English + Italian, test on Roman Urdu
2. Roman Urdu only (full training data)
3. Few-shot: Roman Urdu only vs English + Italian + Roman Urdu, with 100 / 500 / 2000 Roman Urdu examples

Run cells top to bottom.

## 1. Install and check GPU

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
import torch
print('GPU available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change runtime type to T4 GPU')

## 2. Upload your 3 data files
When the button appears, select these 3 files from your laptop (`data/raw` folder):
`english_train.csv`, `italian_train.csv`, `roman_urdu.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

## 3. Load data (same cleaning and same split as Weeks 2 and 3)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_TO_NAME = {0: 'positive', 1: 'neutral', 2: 'negative'}   # English/Italian files use 0/1/2

def load_multilingual(path):
    df = pd.read_csv(path)
    if pd.api.types.is_numeric_dtype(df['label']):
        df['label'] = df['label'].map(NUM_TO_NAME)
    df['text'] = df['text'].astype(str)
    return df[['text', 'label']].dropna().drop_duplicates(subset='text')

en = load_multilingual('english_train.csv')
it = load_multilingual('italian_train.csv')

ru = pd.read_csv('roman_urdu.csv', header=None, usecols=[0, 1],
                 names=['text', 'label'], on_bad_lines='skip').dropna(subset=['text'])
ru['label'] = ru['label'].str.strip().str.lower().replace({'neative': 'negative'})
ru['text'] = ru['text'].astype(str)
ru = ru[ru['text'].str.strip() != ''].drop_duplicates(subset='text')

ru_train, ru_test = train_test_split(ru, test_size=0.2, stratify=ru['label'], random_state=42)
foreign = pd.concat([en, it])
print('English:', len(en), ' Italian:', len(it), ' RU train:', len(ru_train), ' RU test:', len(ru_test))

## 4. Training helper

In [ ]:
import gc, time
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

MODEL = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def to_dataset(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(),
                            'label': [LABEL2ID[l] for l in df['label']]})
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=128),
                  batched=True, remove_columns=['text'])

test_ds = to_dataset(ru_test)

def run(name, train_df, epochs=3):
    set_seed(42)
    t0 = time.time()
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir='out', per_device_train_batch_size=32, per_device_eval_batch_size=64,
        num_train_epochs=epochs, learning_rate=2e-5, weight_decay=0.01,
        fp16=torch.cuda.is_available(), save_strategy='no', report_to='none',
        logging_steps=200, seed=42)
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train_df),
                      data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    logits = trainer.predict(test_ds).predictions
    pred = logits.argmax(axis=-1)
    gold = [LABEL2ID[l] for l in ru_test['label']]
    row = {'experiment': name, 'train_rows': len(train_df),
           'accuracy': round(accuracy_score(gold, pred), 3),
           'macro_f1': round(f1_score(gold, pred, average='macro'), 3),
           'minutes': round((time.time() - t0) / 60, 1)}
    print(row)
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    return row

## 5. Experiment 1 and 2: zero-shot and Roman Urdu only
This takes roughly 10 to 15 minutes on a T4 GPU.

In [ ]:
results = []
results.append(run('zero-shot: EN+IT only', foreign, epochs=3))
results.append(run('Roman Urdu only (full)', ru_train, epochs=3))
pd.DataFrame(results)

## 6. Experiment 3: few-shot (Roman Urdu only vs EN + IT + Roman Urdu)
Small sets get more epochs because there is less data. Takes roughly 15 to 25 minutes.

In [ ]:
for n in [100, 500, 2000]:
    sample = ru_train.sample(n=n, random_state=42)
    epochs = 10 if n <= 100 else 5 if n <= 500 else 4
    results.append(run(f'RU only, {n} examples', sample, epochs=epochs))
    results.append(run(f'EN+IT + {n} RU examples', pd.concat([foreign, sample]), epochs=epochs))

results_df = pd.DataFrame(results)
results_df.to_csv('xlmr_results.csv', index=False)
results_df

## 7. Compare with the TF-IDF baseline (Week 2 and 3)
TF-IDF reference (char n-grams + Linear SVM, same test set): **macro-F1 0.653** with the full Roman Urdu training data, and 0.385 with 50 examples.

In [ ]:
files.download('xlmr_results.csv')   # save this file into your project's docs/ folder

## My notes (write in your own words)
1. **XLM-R zero-shot (EN+IT only) vs TF-IDF zero-shot:** 
2. **XLM-R trained on full Roman Urdu vs TF-IDF (0.653):** 
3. **Does adding EN+IT help XLM-R when Roman Urdu data is small? At which size does it stop helping?** 
4. **What surprised me:** 

In [ ]:
# 1. **Zero-shot result:** English only 0.362, Italian only 0.307, English + Italian
#    0.345 macro-F1. Roman Urdu trained model: 0.653.

# 2. **Close to random (0.33)?** Yes. All three zero-shot scores are within 0.03 of
#    random guessing. Character n-grams from English/Italian words do not match
#    Roman Urdu words, even though the script is the same.

# 3. **Does EN+IT help with small Roman Urdu data?** No. At 100 examples: 0.421 (RU
#    only) vs 0.414 (EN+IT+RU); at 500: 0.517 vs 0.491; at 5000: 0.609 vs 0.605. It
#    slightly hurts below 2500 examples and makes no difference at full size.
#    Roman Urdu alone goes from 0.385 (50 examples) to 0.653 (15,700).

# 4. **Expectation for multilingual transformer:** 